# s23 — Soft-Skills Curator

**What this teaches:** how the harness turns a *finished* session into a reusable playbook. The curator is a one-shot agent that reads a session's audit + state log and decides whether anything is worth distilling into a `softskills/*.md` file the leader can load in future sessions.

**Concept anchor:** [s21_skills](../s21_skills/s21_skills.ipynb) loads playbooks *humans* wrote. Soft-skills are playbooks the *agent itself* wrote based on what it learned. This notebook demonstrates the lifecycle without a live lead agent: write a synthetic session, run the curator, inspect what it produced.

## Prerequisites

- LLM provider configured. Default is `openai_compat` for self-hosted Ollama / vLLM — see [docs/providers.md](../../docs/providers.md).
- The notebook writes to `./tmp-softskills/` in the current directory.
- Without `force=true` the curator may decide nothing is worth curating — that is expected behaviour for trivial sessions.

## 1. Imports

We talk to `softskills` directly — bypassing a live leader. That keeps the demo focused on the curator's contract.

In [ ]:
import (
    "context"
    "fmt"
    "os"
    "path/filepath"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/internal/softskills"
)

## 2. Helper

Panic-based `must`.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. A synthetic 'rich' session

The curator works from two files: an `audit.md` (human-readable narrative) and a `statelog.json` (machine-readable tool-call log). For this demo we synthesize a realistic Docker-OOM debugging session — the kind of multi-step trace the curator should be able to summarise.

In [ ]:
const richAudit = `# Session audit

## Goal
Debug a Docker container that kept restarting with exit code 137 (OOM).

## Actions
1. docker inspect — confirmed OOMKilled=true, RestartCount=12.
2. docker stats — memory hitting 100% of 256 MB limit.
3. docker logs — DEBUG-level JSON log flood causing large allocations.
4. Lowered LOG_LEVEL to INFO via the Compose file; redeployed.
5. Verified memory stabilised at ~90 MB over 3 minutes.

## Outcome
Fixed. Root cause: DEBUG log flood in a memory-constrained container.

## Lesson
When a container keeps OOM-killing, check log volume/verbosity before bumping the memory limit.
`

const richStatelog = `{
  "version": 1,
  "goal": "debug OOM-killed Docker container",
  "outcome": "fixed",
  "tool_calls": [
    {"tool": "bash", "args": "docker inspect <container>"},
    {"tool": "bash", "args": "docker stats --no-stream"},
    {"tool": "bash", "args": "docker logs --tail 200"},
    {"tool": "write", "args": {"path": "docker-compose.yml"}},
    {"tool": "bash", "args": "docker compose up -d"}
  ],
  "decisions": ["log level was DEBUG → changed to INFO"]
}`

tmp, err := os.MkdirTemp("", "s23_session_*")
must(err)
defer os.RemoveAll(tmp)

audit := filepath.Join(tmp, "audit.md")
statelog := filepath.Join(tmp, "statelog.json")
must(os.WriteFile(audit, []byte(richAudit), 0o644))
must(os.WriteFile(statelog, []byte(richStatelog), 0o644))
fmt.Printf("audit:    %s\nstatelog: %s\n", audit, statelog)

## 4. Build the curator runner

`CuratorRunner` returns a ready-to-call ADK runner pre-wired with the curator's instruction and the `write` tool scoped to `SoftSkillsDir`.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

softDir := "tmp-softskills"
r, err := softskills.CuratorRunner(ctx, softskills.CuratorConfig{
    Model:         llm,
    SoftSkillsDir: softDir,
})
must(err)

## 5. Run the curator

`Curate` is the one-shot entry point. It returns the curator's summary paragraph; any soft-skill files it decided to write land in `SoftSkillsDir`.

In [ ]:
out, err := softskills.Curate(ctx, r, softskills.CurateInputs{
    AuditPath:    audit,
    StateLogPath: statelog,
})
must(err)
fmt.Println("── Curator summary ─────────────────────────────────")
fmt.Println(out)
fmt.Println("────────────────────────────────────────────────────")

## 6. Inspect the output directory

Walk the `tmp-softskills/` directory and list whatever the curator produced. Each file is markdown with YAML front matter — same shape as the human-authored skills in [s21_skills](../s21_skills/s21_skills.ipynb).

In [ ]:
fmt.Printf("── %s/ ──\n", softDir)
_ = filepath.Walk(softDir, func(p string, fi os.FileInfo, err error) error {
    if err != nil || fi.IsDir() {
        return err
    }
    rel, _ := filepath.Rel(softDir, p)
    fmt.Printf("  %s\n", rel)
    return nil
})

## What to look for

- The curator's summary either explains what it learned, or politely declines to write anything (trivial sessions yield no soft-skill).
- Compare with [s21_skills](../s21_skills/s21_skills.ipynb): authored skills shape *expected* domains; soft-skills shape *encountered* ones.
- This notebook bypasses the leader; in production the curator runs at session end ([s18_events](../s18_events/s18_events.ipynb) shows the event lifecycle).

## Try it yourself

1. Swap `richAudit` / `richStatelog` for a trivial session (e.g. `echo hello`) and re-run — observe the curator declining.
2. Run the notebook twice with the same `softDir` — the curator should not create duplicate files for the same lesson.